| Student Name | Matric Number |
| :--- | :--- |
| Zhang Jiazheng | A0314707H |
| Phyo Han | A0196680R |
| Wang Shiyu | A0354696L |
| Liu Hengyan | A0350634J |
| Cui Yi | A0353244J |

**Collaborators:** None

**Sources:** None


# Problem 2: Organising Lift Service with an HTN

You now take the capacity-aware lift from Problem 1 into a hierarchical setting. Destinations, the `reached` condition, count objects `c0 ... cC`, and the count-chain invariant retain their earlier meanings. The new challenge is to define an HTN decomposition and select a service order that protects passenger utility.

The guided AI critique from Problem 1 does not continue here. Use any tools you find helpful, but your team remains responsible for validating the HTN and policy; prompt transcripts are neither submitted nor assessed.

## 1. Shared configuration and fixed representation

Each Problem 1 request `(start, goal)` gains three fields describing service quality, while `requests[i]` continues to identify `person{i + 1}`.

```python
config = {
    "num_levels": 5,
    "elevator_start": 2,
    "capacity": 2,
    "requests": [
        {"start": 0, "goal": 4, "deadline": 16,
         "base_utility": 100, "late_penalty": 8},
    ],
}
```

This schema is the complete representation. Hidden cases vary its values and combinations, including shared floors, arbitrary lift starts, initially completed requests, an empty request list, and a one-floor building.

### What carries over from Problem 1

| Problem 1 PDDL | Problem 2 Unified Planning | Meaning |
| --- | --- | --- |
| `person_at(person1, level0)` | `at_person(person1) = floor0` | passenger location is level 0 |
| `person_in_elevator(person1)` | `at_person(person1) = elevator` | passenger is aboard elevator |
| `destination(person1, level4)` | `destination(person1) = floor4` | passenger's destination is level 4 |
| `reached(person1)` | `reached(person1)` | service completed |
| `lift_count(c0)` | `lift_count(c0)` | current occupancy of elevator |
| `next_count(c0, c1)` | `next_count(c0, c1)` | legal increment/decrement |
| `move_up` / `move_down` | `move_elevator(start, end)` | a macro move between floors |

The last row is an abstraction: one HTN move may span several floors, while its external duration will still be one time unit per floor crossed, or `abs(start - end)`.

In [1]:
import importlib.util
from pathlib import Path

def load_submission(path="2_hierarchical_solution.py"):
    spec = importlib.util.spec_from_file_location("problem2_submission", Path(path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

student = load_submission()

In [2]:
config = {
    "num_levels": 5,
    "elevator_start": 2,
    "capacity": 2,
    "requests": [
        {"start": 0, "goal": 4, "deadline": 16, "base_utility": 100, "late_penalty": 8},
        {"start": 3, "goal": 1, "deadline": 12, "base_utility": 80, "late_penalty": 12},
        {"start": 2, "goal": 2, "deadline": 0, "base_utility": 30, "late_penalty": 5},
    ],
}
student.validate_config(config)

## 2. Deterministic capacity batches

A service order is a permutation of **all** passenger-request indices. Requests are then executed in deterministic batches. Passengers already at their destinations need neither pickup nor delivery: they are removed before batching and confirmed at the end. The active passengers are divided into consecutive groups containing at most `capacity` entries.

Each active batch executes strictly in sequence:

1. pick up every passenger in batch order;
2. deliver every passenger in the same order;
3. only then begin the next batch.

For example, `capacity = 2` and service order `[2, 0, 3, 1]`, where passenger 2 is already at the goal, the active batches are `[[0, 3], [1]]`. The elevator executes: **pickup 0 $\to$ pickup 3 $\to$ deliver 0 $\to$ deliver 3 $\to$ pickup 1 $\to$ deliver 1 $\to$ confirm 2**.

This fixed protocol makes different policies comparable and makes capacity observable in the primitive plan.

## Task 1: Design the service-order policy (2 points)

1. Open the file: `2_hierarchical_solution.py`
2. Complete `choose_service_order` inside `COPY-FLAG-1` of the python script. The starter order is valid but generally poor.
3. An external evaluator scores orders using these rules:
    - **Travel time:** Moving between floors `a` and `b` takes `abs(a - b)` time units.
    - **Action duration:** Opening, closing, loading, and unloading take 1 time unit each.
    - **Passenger completion:** Recorded immediately after unloading (passengers who are initially already at destination complete at $t = 0$).
    - **Passenger utility:** `utility = max(0, base_utility - late_penalty * max(0, completion - deadline))`.

*Note: Travel distance and finish time are reported for analysis, but your marks depend strictly on total utility. For a quality case to receive utility credit, the submitted HTN must also build the required task network, return a plan, and pass independent replay; otherwise that case receives zero quality credit. Your policy must generalize; the visible notebook examples are solely for debugging.*

### Grading & Constraints

There are ten equally weighted quality configurations with capacities from 1 through 3. Their active-passenger distribution is: 1--5 passengers (20%), 6--7 (20%), 8--9 (30%), and 10 (30%). Each policy call runs in a separate process with a hard 10-second limit; an overrun is terminated. Your policy must generalize; the visible configuration is not a grading instance.

In [3]:
student = load_submission()  # reload after editing the .py file
order = student.choose_service_order(config)
print("order:", order)
print("batches:", student.service_batches(config, order))
print("evaluation:", student.evaluate_service_order(config, order))

order: [1, 0, 2]
batches: [[1, 0]]
evaluation: {'order': [1, 0, 2], 'batches': [[1, 0]], 'completion_times': [19, 13, 0], 'utilities': [76, 68, 30], 'total_utility': 174, 'sum_completion_time': 32, 'max_completion_time': 19, 'total_travel': 8, 'finish_time': 20}


### Additional public policy cases

The three cases below are merely for debugging. They give you small, reproducible checks for capacity one, a partial final batch with an initially reached passenger, and shared floors. The published optimum is the maximum **total utility**; your current policy may score below it while you are developing.

In [4]:
public_quality_cases = [
    {
        "name": "capacity-one urgency",
        "optimal_utility": 214,
        "config": {
            "num_levels": 6, "elevator_start": 2, "capacity": 1,
            "requests": [
                {"start": 0, "goal": 5, "deadline": 20, "base_utility": 100, "late_penalty": 8},
                {"start": 4, "goal": 1, "deadline": 12, "base_utility": 80, "late_penalty": 12},
                {"start": 2, "goal": 3, "deadline": 8, "base_utility": 60, "late_penalty": 10},
                {"start": 5, "goal": 0, "deadline": 25, "base_utility": 90, "late_penalty": 6},
            ],
        },
    },
    {
        "name": "partial batch and initially reached",
        "optimal_utility": 287,
        "config": {
            "num_levels": 7, "elevator_start": 3, "capacity": 2,
            "requests": [
                {"start": 0, "goal": 6, "deadline": 26, "base_utility": 120, "late_penalty": 7},
                {"start": 5, "goal": 1, "deadline": 17, "base_utility": 95, "late_penalty": 9},
                {"start": 2, "goal": 4, "deadline": 12, "base_utility": 70, "late_penalty": 10},
                {"start": 3, "goal": 3, "deadline": 0, "base_utility": 25, "late_penalty": 3},
                {"start": 6, "goal": 0, "deadline": 24, "base_utility": 110, "late_penalty": 8},
                {"start": 1, "goal": 5, "deadline": 19, "base_utility": 85, "late_penalty": 7},
            ],
        },
    },
    {
        "name": "shared floors, capacity three",
        "optimal_utility": 254,
        "config": {
            "num_levels": 8, "elevator_start": 0, "capacity": 3,
            "requests": [
                {"start": 1, "goal": 7, "deadline": 24, "base_utility": 115, "late_penalty": 8},
                {"start": 1, "goal": 5, "deadline": 18, "base_utility": 90, "late_penalty": 10},
                {"start": 6, "goal": 2, "deadline": 21, "base_utility": 100, "late_penalty": 9},
                {"start": 6, "goal": 0, "deadline": 20, "base_utility": 95, "late_penalty": 9},
                {"start": 3, "goal": 5, "deadline": 14, "base_utility": 75, "late_penalty": 11},
                {"start": 7, "goal": 1, "deadline": 27, "base_utility": 105, "late_penalty": 7},
            ],
        },
    },
]

student = load_submission()
for case in public_quality_cases:
    case_config = case["config"]
    order = student.choose_service_order(case_config)
    evaluation = student.evaluate_service_order(case_config, order)
    achieved = evaluation["total_utility"]
    optimum = case["optimal_utility"]
    assert achieved <= optimum, "utility exceeds the verified optimum; check the evaluator"
    print(f"{case['name']}: utility={achieved}/{optimum}, gap={optimum - achieved}, order={order}")

capacity-one urgency: utility=214/214, gap=0, order=[2, 0, 3, 1]
partial batch and initially reached: utility=287/287, gap=0, order=[1, 4, 0, 2, 5, 3]
shared floors, capacity three: utility=254/254, gap=0, order=[0, 2, 5, 3, 4, 1]


### Search-strategy interlude: BFS and DFS (ungraded)

View service-order construction as a tree whose nodes are partial permutations and whose children append one unserved passenger. The experiment below alters only the frontier rule: breadth-first search takes the oldest node, whereas depth-first search takes the newest. It is an algorithm-design aid, contributes no marks, and supplies no submission code.

In [5]:
from collections import deque

def compare_order_search(passenger_count, strategy):
    frontier = deque([((), tuple(range(passenger_count)))])
    expanded = 0
    max_frontier = 1
    first_complete_at = None

    while frontier:
        max_frontier = max(max_frontier, len(frontier))
        if strategy == "bfs":
            partial, remaining = frontier.popleft()
        elif strategy == "dfs":
            partial, remaining = frontier.pop()
        else:
            raise ValueError("strategy must be 'bfs' or 'dfs'")

        expanded += 1
        if not remaining:
            if first_complete_at is None:
                first_complete_at = expanded
            continue

        for passenger in remaining:
            next_remaining = tuple(x for x in remaining if x != passenger)
            frontier.append((partial + (passenger,), next_remaining))

    return {
        "strategy": strategy,
        "expanded": expanded,
        "max_frontier": max_frontier,
        "first_complete_at": first_complete_at,
    }

for passenger_count in (4, 6, 8):
    for strategy in ("bfs", "dfs"):
        print(passenger_count, compare_order_search(passenger_count, strategy))

4 {'strategy': 'bfs', 'expanded': 65, 'max_frontier': 24, 'first_complete_at': 42}
4 {'strategy': 'dfs', 'expanded': 65, 'max_frontier': 7, 'first_complete_at': 5}
6 {'strategy': 'bfs', 'expanded': 1957, 'max_frontier': 720, 'first_complete_at': 1238}
6 {'strategy': 'dfs', 'expanded': 1957, 'max_frontier': 16, 'first_complete_at': 7}
8 {'strategy': 'bfs', 'expanded': 109601, 'max_frontier': 40320, 'first_complete_at': 69282}
8 {'strategy': 'dfs', 'expanded': 109601, 'max_frontier': 29, 'first_complete_at': 9}


**Key observation.** All complete service orders occur at the same depth, so BFS's shortest-path property says nothing about utility. DFS finds a complete candidate sooner and stores a smaller frontier, yet exhaustive versions of BFS and DFS still enumerate the same partial permutations.

When designing the policy, ask how a strong incumbent can enable pruning and when two prefixes have an equivalent future. Determine which information, beyond the served-passenger set, fixes every remaining cost. Those questions point toward branch-and-bound or dynamic programming rather than a frontier swap alone.

## 3. Decompose the service into an HTN

The representation fixes five primitive actions: `move_elevator`, `load`, `unload`, `open_door`, and `close_door`. A load advances exactly one count link and excludes completed passengers; an unload verifies the destination, sets `reached`, and moves exactly one link downward.

Three abstract tasks organise those actions:

- `pickup_person(person, start_floor)`;
- `deliver_person(person, goal_floor)`;
- `confirm_reached(person, goal_floor)`.

Pickup and delivery each require two methods to distinguish whether the lift is already on the required floor. The `confirm_reached` method decomposes into no subtasks.

## Task 2: Complete the five primitive actions (1 point)

Inside `COPY-FLAG-2`, implement the fixed definitions below. The grader compares them exactly, so do not insert or remove guards. The template already supplies every parameter signature.

| Action | Exact preconditions | Exact effects |
| --- | --- | --- |
| `move_elevator` | elevator at `start`; door closed | elevator at `end` |
| `load` | elevator and person at `floor`; door open; `lift_count(current)`; `next_count(current,next)`; person not reached | person at elevator; current count false; next count true |
| `unload` | elevator at `floor`; person at elevator; door open; person's destination is `floor`; `lift_count(current)`; `next_count(previous,current)` | person at floor; reached true; current count false; previous count true |
| `open_door` | door closed | door open |
| `close_door` | door open | door closed |

In particular, movement does not require a separate `start != end` guard, and unload does not require an additional `not reached` guard.

## Task 3: Complete exactly five methods (1 point)

1. Open the file: `2_hierarchical_solution.py`
2. Inside `COPY-FLAG-3`, use these exact parameter signatures, in the shown order:

    - `method_pickup_from_other_floor(elevator, person, elevator_floor, start_floor, current, next)`;
    - `method_pickup_from_current_floor(elevator, person, start_floor, current, next)`;
    - `method_deliver_to_other_floor(elevator, person, elevator_floor, goal_floor, previous, current)`;
    - `method_deliver_at_current_floor(elevator, person, goal_floor, previous, current)`;
    - `method_confirm_reached(person, goal_floor)`.

Pickup methods check the person's start, the elevator floor, a closed door, the applicable count link, and that the passenger is not reached. The `other_floor` method additionally checks that the two floors differ. Delivery methods analogously check that the person is in the elevator, the destination, elevator floor, closed door, and the decrementing count link; the `other_floor` method checks that the two floors differ. `method_confirm_reached` checks reached, location, and destination.

Use the exact ordered decompositions `move/open/load/close`, `open/load/close`, `move/open/unload/close`, `open/unload/close`, and the empty decomposition, respectively. Do not add alternative actions, methods, tasks, fluents, or extra preconditions.

## Task 4: Build the ordered task network (1 point; solvability and independent replay are another 3 points)

1. Open the file: `2_hierarchical_solution.py`
2. Inside `COPY-FLAG-4`, encode the selected order using the fixed batching protocol. For each active batch, add all pickups and then all deliveries. Append one `confirm_reached` task for every initially reached passenger, in their relative policy order. The complete initial network must be totally ordered.

In [6]:
student = load_submission()
problem = student.generate_hierarchical(config)
print(problem.task_network)
result = student.solve(problem, verbose=True)
assert result.plan is not None

task network {
  subtasks = [
    _t15: pickup_person(person2, floor3)
    _t16: pickup_person(person1, floor0)
    _t17: deliver_person(person2, floor1)
    _t18: deliver_person(person1, floor4)
    _t19: confirm_reached(person3, floor2)
  ]
  constraints = [
    (end(_t15) < start(_t16))
    (end(_t16) < start(_t17))
    (end(_t17) < start(_t18))
    (end(_t18) < start(_t19))
  ]
}
Plan: # Action Plan
SequentialPlan:
    move_elevator(elevator, floor2, floor3)
    open_door(elevator)
    load(elevator, person2, floor3, c0, c1)
    close_door(elevator)
    move_elevator(elevator, floor3, floor0)
    open_door(elevator)
    load(elevator, person1, floor0, c1, c2)
    close_door(elevator)
    move_elevator(elevator, floor0, floor1)
    open_door(elevator)
    unload(elevator, person2, floor1, c1, c2)
    close_door(elevator)
    move_elevator(elevator, floor1, floor4)
    open_door(elevator)
    unload(elevator, person1, floor4, c0, c1)
    close_door(elevator)
# Decomposition
_t15 -> m

## 4. Exercise boundary cases before submission

Test every case below rather than relying solely on the worked example:

- `capacity == 1`;
- more active passengers than capacity, including a partial final batch;
- several passengers sharing starts or goals;
- a pickup or delivery already at the elevator's current floor;
- one or more initially reached passengers mixed with active passengers;
- no passengers and a one-floor building.

Obtaining a plan is only the first check. Confirm that its primitive actions obey pickup-all/deliver-all batching, respect capacity, unload passengers only at their destinations, finish empty with the door closed, and reproduce the evaluator's completion times.

## Submission and grading (8 points)

Submit `2_hierarchical_solution.py`. Leave all supplied code outside the four copy flags untouched. The notebook itself is not graded. The 8 points comprise primitive actions 1, HTN methods 1, ordered task network 1, solvability and independent replay 3, and policy quality 2. Replay credit is averaged across the published configuration classes rather than awarded all at once. For each quality case, `U_min` and `U_max` are the exact minimum and maximum total utilities over all valid passenger permutations, and `q = clip((U_student - U_min) / (U_max - U_min), 0, 1)` (or `q = 1` when the range is zero), with quality worth `2 * mean(q)`. A case receives zero quality credit if its HTN, task network, solve, or independent replay fails. Because the grader replays the primitive plan independently, editing the supplied evaluator cannot alter the mark.